<a href="https://colab.research.google.com/github/TralAlex/IW/blob/main/PRVI_zad_Copy_of_Big_Data_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Projektni Zadatak: Razvoj Sistema Preporuke u
PySpark-u

1. Najpopularniji filmovi

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Putanja do foldera u Drive-u
dataset_path = "/content/drive/MyDrive/ml-100k"

# Provera da li Spark vidi fajlove
import os
print(os.listdir(dataset_path))
print(os.path.exists(f"{dataset_path}/u.data"))

Mounted at /content/drive
['u1.base', 'u1.test', 'ua.base', 'ua.test', 'u.item', 'u.info', 'u.data', 'mku.sh', 'u.occupation', 'ub.test', 'ub.base', 'u.genre', 'allbut.pl', 'README', 'u.user', 'u3.test', 'u2.test', 'u2.base', 'u3.base', 'u4.base', 'u5.base', 'u4.test', 'u5.test']
True


In [ ]:
# Conenct to Spark
# SparkSession predstavlja ulaznu tačku za rad sa Apache Spark-om.
# Omogućava kreiranje DataFrame-ova, izvršavanje SQL upita i upravljanje Spark okruženjem.
# Uvodimo ga kako bismo mogli da radimo distribuiranu obradu velikih skupova podataka.
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("MovieLensAnalysis").getOrCreate()

In [ ]:
# Učitavanje dataset-a ocena u Spark DataFrame iz CSV fajla,imamo separator.
# Preimenujemo kolone radi lakšeg rada i čitljivosti modela.
ratings = spark.read.csv(
    f"{dataset_path}/u.data",
    sep="\t",
    inferSchema=True,
    header=False
)

ratings = ratings.toDF("user_id", "movie_id", "rating", "timestamp")
ratings.show(5)

+-------+--------+------+---------+
|user_id|movie_id|rating|timestamp|
+-------+--------+------+---------+
|    196|     242|     3|881250949|
|    186|     302|     3|891717742|
|     22|     377|     1|878887116|
|    244|      51|     2|880606923|
|    166|     346|     1|886397596|
+-------+--------+------+---------+
only showing top 5 rows


In [ ]:
# Učitavanje podataka o filmovima iz CSV fajla sa "|" separatorom.
# Dodajemo encoding jer dataset može sadržati specijalne karaktere u nazivima filmova.
# Biramo samo relevantne kolone (ID i naziv filma) i preimenujemo ih.
# Prikazujemo prvih nekoliko redova bez skraćivanja teksta radi provere.

movies = spark.read.csv(
    f"{dataset_path}/u.item",
    sep="|",
    inferSchema=True,
    header=False,
    encoding="ISO-8859-1"
)

movies = movies.selectExpr("_c0 as movie_id", "_c1 as title")

movies.show(5, truncate=False)

+--------+-----------------+
|movie_id|title            |
+--------+-----------------+
|1       |Toy Story (1995) |
|2       |GoldenEye (1995) |
|3       |Four Rooms (1995)|
|4       |Get Shorty (1995)|
|5       |Copycat (1995)   |
+--------+-----------------+
only showing top 5 rows


In [ ]:
# Učitavanje korisnicima,sep "|" ,enkoding spec karaktera.
# Novi nazivi kolona.
# Podaci će se koristiti za analizu korisničkih karakteristika i preporuke.

users = spark.read.csv(
    f"{dataset_path}/u.user",
    sep="|",
    inferSchema=True,
    header=False,
    encoding="ISO-8859-1"
)

users = users.toDF("user_id", "age", "gender", "occupation", "zip_code")

users.show(5)

+-------+---+------+----------+--------+
|user_id|age|gender|occupation|zip_code|
+-------+---+------+----------+--------+
|      1| 24|     M|technician|   85711|
|      2| 53|     F|     other|   94043|
|      3| 23|     M|    writer|   32067|
|      4| 24|     M|technician|   43537|
|      5| 33|     F|     other|   15213|
+-------+---+------+----------+--------+
only showing top 5 rows


- Prikazati 50 najpopularnijih filmova.

In [ ]:
from pyspark.sql.functions import count, avg, round, max, col, row_number, array, when, explode

#spoj ocene i filmova preko movie_id.

movie_ratings = ratings.join(movies, on="movie_id", how="inner")

# Grupisanjem po filmu brojimo koliko puta je svaki film ocenjen.
# Kolona "views" predstavlja broj ocena, odnosno popularnost filma u dataset-u.

popular_movies = movie_ratings.groupBy("movie_id", "title").agg(
    count("*").alias("views")
).orderBy("views", ascending=False)

# Prikazujemo 50 najpopularnijih filmova bez skraćivanja naziva.
popular_movies.show(50, truncate=False)

+--------+--------------------------------------------+-----+
|movie_id|title                                       |views|
+--------+--------------------------------------------+-----+
|50      |Star Wars (1977)                            |583  |
|258     |Contact (1997)                              |509  |
|100     |Fargo (1996)                                |508  |
|181     |Return of the Jedi (1983)                   |507  |
|294     |Liar Liar (1997)                            |485  |
|286     |English Patient, The (1996)                 |481  |
|288     |Scream (1996)                               |478  |
|1       |Toy Story (1995)                            |452  |
|300     |Air Force One (1997)                        |431  |
|121     |Independence Day (ID4) (1996)               |429  |
|174     |Raiders of the Lost Ark (1981)              |420  |
|127     |Godfather, The (1972)                       |413  |
|56      |Pulp Fiction (1994)                         |394  |
|7      

- Za svaki film izračunati broj pregleda i prosečnu ocenu i prikazati nazive 10 filmova sa
najviše i 10 filmova sa najmanje pregleda.

In [ ]:
# Grupisanjem po filmu računamo:
# - broj ocena (views) kao meru popularnosti
# - prosečnu ocenu (avg_rating) kao meru kvaliteta
movie_stats = movie_ratings.groupBy("movie_id", "title").agg(
    count("*").alias("views"),
    round(avg("rating"), 2).alias("avg_rating")
)

# Top 10 najpopularnijih filmova (najviše ocena)
print("Top 10 movies with the most views:")
movie_stats.orderBy("views", ascending=False).show(10, truncate=False)

# Top 10 najmanje gledanih filmova (najmanje ocena)
print("Top 10 movies with the least views:")
movie_stats.orderBy("views", ascending=True).show(10, truncate=False)

Top 10 movies with the most views:
+--------+-----------------------------+-----+----------+
|movie_id|title                        |views|avg_rating|
+--------+-----------------------------+-----+----------+
|50      |Star Wars (1977)             |583  |4.36      |
|258     |Contact (1997)               |509  |3.8       |
|100     |Fargo (1996)                 |508  |4.16      |
|181     |Return of the Jedi (1983)    |507  |4.01      |
|294     |Liar Liar (1997)             |485  |3.16      |
|286     |English Patient, The (1996)  |481  |3.66      |
|288     |Scream (1996)                |478  |3.44      |
|1       |Toy Story (1995)             |452  |3.88      |
|300     |Air Force One (1997)         |431  |3.63      |
|121     |Independence Day (ID4) (1996)|429  |3.44      |
+--------+-----------------------------+-----+----------+
only showing top 10 rows
Top 10 movies with the least views:
+--------+-----------------------------------------------+-----+----------+
|movie_id|title 

- Izračunati “skor” za svaki film na osnovu sledeće formule:
(broj_pregleda*prosečna_ocena)/(max(broj_pregleda)*max(prosečna ocena)). Prikazati top
5 filmova na osnovu izračunatog „skora“.

In [ ]:
# Pronalazimo maksimalne vrednosti za normalizaciju
max_views = movie_stats.agg(max("views")).collect()[0][0]
max_avg_rating = movie_stats.agg(max("avg_rating")).collect()[0][0]

# Kreiramo skor koji kombinuje popularnost i prosečnu ocenu
movie_scores = movie_stats.withColumn(
    "score",
    round((col("views") * col("avg_rating")) / (max_views * max_avg_rating), 4)
)

# Prikaz top 5 filmova po skoru
movie_scores.orderBy("score", ascending=False).show(5, truncate=False)
# Normalizujemo vrednosti kako bismo mogli da kombinujemo različite metrike.
# Score predstavlja balans između popularnosti (views) i kvaliteta (avg_rating).
# Viši score znači da je film i gledan i visoko ocenjen.

+--------+------------------------------+-----+----------+------+
|movie_id|title                         |views|avg_rating|score |
+--------+------------------------------+-----+----------+------+
|50      |Star Wars (1977)              |583  |4.36      |0.872 |
|100     |Fargo (1996)                  |508  |4.16      |0.725 |
|181     |Return of the Jedi (1983)     |507  |4.01      |0.6975|
|258     |Contact (1997)                |509  |3.8       |0.6635|
|174     |Raiders of the Lost Ark (1981)|420  |4.25      |0.6123|
+--------+------------------------------+-----+----------+------+
only showing top 5 rows


- Prikazati 5 najpopularnijih filmova po polu.

In [ ]:
# Import samo Window (ostalo je već uvezeno ranije)
from pyspark.sql.window import Window
# Spajamo prethodno dobijene podatke o ocenama i filmovima sa podacima o korisnicima.
# Na ovaj način svaka ocena dobija i informaciju o polu korisnika.
movie_ratings_users = movie_ratings.join(users, on="user_id", how="inner")

# Računamo popularnost filmova po polu korisnika.
# Views predstavlja broj ocena koje je određeni film dobio u okviru svake gender grupe.
gender_popularity = movie_ratings_users.groupBy("gender", "title").agg(
    count("*").alias("views")
)

# Definišemo window po polu korisnika i sortiramo filmove po broju ocena opadajuće.
# Ovo omogućava rangiranje filmova posebno za svaku gender grupu.
window_spec_gender = Window.partitionBy("gender").orderBy(col("views").desc())

# Dodeljujemo rang svakom filmu unutar njegove gender grupe.
ranked_movies_gender = gender_popularity.withColumn(
    "rank",
    row_number().over(window_spec_gender)
)

# Izdvajamo top 5 najpopularnijih filmova za svaki pol korisnika.
top_5_by_gender = ranked_movies_gender.filter(col("rank") <= 5)

# Prikaz rezultata sortiranih po polu i rangu.
top_5_by_gender.orderBy("gender", "rank").show(truncate=False)

+------+---------------------------+-----+----+
|gender|title                      |views|rank|
+------+---------------------------+-----+----+
|F     |English Patient, The (1996)|152  |1   |
|F     |Star Wars (1977)           |151  |2   |
|F     |Scream (1996)              |143  |3   |
|F     |Liar Liar (1997)           |141  |4   |
|F     |Contact (1997)             |137  |5   |
|M     |Star Wars (1977)           |432  |1   |
|M     |Return of the Jedi (1983)  |383  |2   |
|M     |Fargo (1996)               |383  |3   |
|M     |Contact (1997)             |372  |4   |
|M     |Liar Liar (1997)           |344  |5   |
+------+---------------------------+-----+----+



- Prikazati 3 najpopularnija filma po polu i zanimanju korisnika.

In [ ]:
# Računamo popularnost filmova po kombinaciji pola i zanimanja korisnika.
# Na ovaj način dobijamo detaljniju segmentaciju korisnika.
gender_occupation_popularity = movie_ratings_users.groupBy("gender", "occupation", "title").agg(
    count("*").alias("views")
)

# Definišemo window po polu i zanimanju, sortirano po broju ocena opadajuće.
# Ovo omogućava rangiranje filmova unutar svake (gender, occupation) grupe.
window_spec_gender_occupation = Window.partitionBy("gender", "occupation").orderBy(col("views").desc())

# Dodeljujemo rang svakom filmu unutar njegove grupe.
ranked_movies_gender_occupation = gender_occupation_popularity.withColumn(
    "rank",
    row_number().over(window_spec_gender_occupation)
)

# Izdvajamo top 3 najpopularnija filma za svaku kombinaciju pola i zanimanja.
top_3_by_gender_and_occupation = ranked_movies_gender_occupation.filter(col("rank") <= 3)

# Prikaz rezultata sortiranih po polu, zanimanju i rangu.
top_3_by_gender_and_occupation.orderBy("gender", "occupation", "rank").show(truncate=False)

+------+-------------+------------------------------------+-----+----+
|gender|occupation   |title                               |views|rank|
+------+-------------+------------------------------------+-----+----+
|F     |administrator|English Patient, The (1996)         |21   |1   |
|F     |administrator|Star Wars (1977)                    |21   |2   |
|F     |administrator|Jerry Maguire (1996)                |19   |3   |
|F     |artist       |Contact (1997)                      |10   |1   |
|F     |artist       |Godfather, The (1972)               |8    |2   |
|F     |artist       |Chasing Amy (1997)                  |8    |3   |
|F     |educator     |English Patient, The (1996)         |23   |1   |
|F     |educator     |Contact (1997)                      |15   |2   |
|F     |educator     |Full Monty, The (1997)              |14   |3   |
|F     |engineer     |Evita (1996)                        |2    |1   |
|F     |engineer     |English Patient, The (1996)         |2    |2   |
|F    

- Prikazati 3 najpopularnija filma u svakom žanru.

In [ ]:
# Učitavamo kompletan u.item fajl jer pored naziva filma sadrži i žanrove.
# Encoding dodajemo zbog specijalnih karaktera u nazivima filmova.
movies_raw = spark.read.csv(
    f"{dataset_path}/u.item",
    sep="|",
    inferSchema=True,
    header=False,
    encoding="ISO-8859-1"
)

# Dodeljujemo nazive kolonama prema MovieLens strukturi.
movies_full = movies_raw.toDF(
    "movie_id", "title", "release_date", "video_release_date",
    "imdb_url", "unknown", "Action", "Adventure", "Animation",
    "Children", "Comedy", "Crime", "Documentary", "Drama",
    "Fantasy", "FilmNoir", "Horror", "Musical", "Mystery",
    "Romance", "SciFi", "Thriller", "War", "Western"
)

# Kreiramo niz žanrova za svaki film na osnovu binarnih kolona.
movies_genres = movies_full.select(
    "movie_id",
    "title",
    array(
        when(col("Action") == 1, "Action"),
        when(col("Adventure") == 1, "Adventure"),
        when(col("Animation") == 1, "Animation"),
        when(col("Children") == 1, "Children"),
        when(col("Comedy") == 1, "Comedy"),
        when(col("Crime") == 1, "Crime"),
        when(col("Documentary") == 1, "Documentary"),
        when(col("Drama") == 1, "Drama"),
        when(col("Fantasy") == 1, "Fantasy"),
        when(col("FilmNoir") == 1, "Film-Noir"),
        when(col("Horror") == 1, "Horror"),
        when(col("Musical") == 1, "Musical"),
        when(col("Mystery") == 1, "Mystery"),
        when(col("Romance") == 1, "Romance"),
        when(col("SciFi") == 1, "Sci-Fi"),
        when(col("Thriller") == 1, "Thriller"),
        when(col("War") == 1, "War"),
        when(col("Western") == 1, "Western")
    ).alias("genres")
)

# Explode pretvara niz žanrova u više redova, tako da jedan film može pripadati više žanrova.
movies_by_genre = movies_genres.withColumn(
    "genre",
    explode(col("genres"))
).filter(col("genre").isNotNull())

# Spajamo ocene sa filmovima i njihovim žanrovima.
movie_with_ratings = ratings.join(movies_by_genre, on="movie_id", how="inner")

# Računamo popularnost filmova unutar svakog žanra.
genre_popularity = movie_with_ratings.groupBy("genre", "title").agg(
    count("*").alias("views")
)

# Rangiramo filmove posebno unutar svakog žanra prema broju ocena.
window_spec_genre = Window.partitionBy("genre").orderBy(col("views").desc())

ranked_movies_genre = genre_popularity.withColumn(
    "rank",
    row_number().over(window_spec_genre)
)

# Izdvajamo top 3 najpopularnija filma za svaki žanr.
top_3_genres = ranked_movies_genre.filter(col("rank") <= 3)

# Prikaz rezultata sortiranih po žanru i rangu.
top_3_genres.orderBy("genre", "rank").show(truncate=False)

+-----------+--------------------------------------------+-----+----+
|genre      |title                                       |views|rank|
+-----------+--------------------------------------------+-----+----+
|Action     |Star Wars (1977)                            |583  |1   |
|Action     |Return of the Jedi (1983)                   |507  |2   |
|Action     |Air Force One (1997)                        |431  |3   |
|Adventure  |Star Wars (1977)                            |583  |1   |
|Adventure  |Return of the Jedi (1983)                   |507  |2   |
|Adventure  |Raiders of the Lost Ark (1981)              |420  |3   |
|Animation  |Toy Story (1995)                            |452  |1   |
|Animation  |Lion King, The (1994)                       |220  |2   |
|Animation  |Aladdin (1992)                              |219  |3   |
|Children   |Toy Story (1995)                            |452  |1   |
|Children   |Willy Wonka and the Chocolate Factory (1971)|326  |2   |
|Children   |E.T. th

- Kreirati atribut kategorički starosna_grupa na osnovu atributa age i prikazati 5
najpopularnijih filmova za svaku grupu.

In [ ]:
# Kreiramo novu kolonu "starosna_grupa" kako bismo segmentirali korisnike po godinama.
# Segmentacija omogućava analizu preferencija različitih starosnih grupa.
data = movie_ratings_users.withColumn(
    "starosna_grupa",
    when(col("age") <= 17, "teen")
    .when((col("age") >= 18) & (col("age") <= 25), "young-adult")
    .when((col("age") >= 26) & (col("age") <= 50), "adult")
    .otherwise("senior")
)

# Računamo popularnost filmova unutar svake starosne grupe.
grouped_starosna_grupa = data.groupBy("starosna_grupa", "title").agg(
    count("*").alias("views")
)

# Definišemo window po starosnoj grupi i sortiramo po broju ocena opadajuće.
window_spec_starosna_grupa = Window.partitionBy("starosna_grupa").orderBy(col("views").desc())

# Dodeljujemo rang filmovima unutar svake starosne grupe.
ranked_starosna_grupa = grouped_starosna_grupa.withColumn(
    "rank",
    row_number().over(window_spec_starosna_grupa)
)

# Izdvajamo top 5 filmova za svaku starosnu grupu.
top_5_age = ranked_starosna_grupa.filter(col("rank") <= 5)

# Prikaz rezultata sortiranih po starosnoj grupi i rangu.
top_5_age.orderBy("starosna_grupa", "rank").show(truncate=False)

+--------------+---------------------------+-----+----+
|starosna_grupa|title                      |views|rank|
+--------------+---------------------------+-----+----+
|adult         |Star Wars (1977)           |362  |1   |
|adult         |Fargo (1996)               |314  |2   |
|adult         |Return of the Jedi (1983)  |307  |3   |
|adult         |Contact (1997)             |303  |4   |
|adult         |English Patient, The (1996)|300  |5   |
|senior        |English Patient, The (1996)|80   |1   |
|senior        |Fargo (1996)               |60   |2   |
|senior        |Air Force One (1997)       |54   |3   |
|senior        |Full Monty, The (1997)     |50   |4   |
|senior        |Godfather, The (1972)      |48   |5   |
|teen          |Scream (1996)              |28   |1   |
|teen          |Liar Liar (1997)           |21   |2   |
|teen          |Contact (1997)             |20   |3   |
|teen          |Star Wars (1977)           |19   |4   |
|teen          |Return of the Jedi (1983)  |18  

- Prikazati ukupnu prosečnu ocenu svih filmova.

In [ ]:
# Računamo globalnu prosečnu ocenu svih filmova u dataset-u.
# Ova vrednost predstavlja osnovnu referencu (baseline) za poređenje sa pojedinačnim filmovima ili modelima.
overall_avg_ratings = ratings.agg(
    round(avg("rating"), 2).alias("global_avg_rating")
)

# Prikaz rezultata
overall_avg_ratings.show()

+-----------------+
|global_avg_rating|
+-----------------+
|             3.53|
+-----------------+



2. Sistem preporuke baziran na sadržaju

- Koristeći attribute age, gender, occupation, genre i movie year (ovaj atribut je potrebno
ekstrahovati iz naziva filma) i izlaznu varijablu ratings kreirati regresioni model korišćenjem
Random forest i Linearne regresije. Za svaki algoritam optimizujte hiper-parametre
korišćenjem Cross-validacije. Za trening skup koristite u1.base a kao test skup u1.test.

**Content-based** filtering preporučuje slične stavke na osnovu osobina sadržaja koje je korisnik ranije voleo (žanr, ključne reči, glumci itd.).

Prednosti: ne zavisi od drugih korisnika, dobre personalizovane preporuke.
Content-Based Filtering koristi **karakteristike sadržaja **i **item-item sličnosti za preporuku sličnih stavki koje je korisnik ranije voleo.**

In [ ]:
# Import funkcije za rad sa tekstom i kolonama
from pyspark.sql.functions import regexp_extract, col, when

# Izvlačimo godinu iz naziva filma (npr. "Toy Story (1995)")
movies_year = movies_full.withColumn(
    "year_str",
    regexp_extract(col("title"), r"\((\d{4})\)", 1)
).withColumn(
    "year",
    when(col("year_str") != "", col("year_str").cast("int")).otherwise(None)
)

# Dodeljujemo jedan žanr po filmu (prvi koji je označen kao 1)
# Napomena: film može imati više žanrova, ali ovde uzimamo samo jedan (po redu kolona)
movies_features = movies_year.withColumn(
    "genre",
    when(col("Action") == 1, "Action")
    .when(col("Adventure") == 1, "Adventure")
    .when(col("Animation") == 1, "Animation")
    .when(col("Children") == 1, "Children")
    .when(col("Comedy") == 1, "Comedy")
    .when(col("Crime") == 1, "Crime")
    .when(col("Documentary") == 1, "Documentary")
    .when(col("Drama") == 1, "Drama")
    .when(col("Fantasy") == 1, "Fantasy")
    .when(col("FilmNoir") == 1, "Film-Noir")
    .when(col("Horror") == 1, "Horror")
    .when(col("Musical") == 1, "Musical")
    .when(col("Mystery") == 1, "Mystery")
    .when(col("Romance") == 1, "Romance")
    .when(col("SciFi") == 1, "Sci-Fi")
    .when(col("Thriller") == 1, "Thriller")
    .when(col("War") == 1, "War")
    .when(col("Western") == 1, "Western")
)

# Spajamo sve podatke u jedinstven dataset za modelovanje
model_data = ratings.join(users, on="user_id") \
    .join(movies_features.select("movie_id", "genre", "year"), on="movie_id") \
    .select("age", "gender", "occupation", "genre", "year", "rating")

# Prikaz rezultata
model_data.show(5, truncate=False)

+---+------+----------+--------+----+------+
|age|gender|occupation|genre   |year|rating|
+---+------+----------+--------+----+------+
|49 |M     |writer    |Comedy  |1996|3     |
|39 |F     |executive |Crime   |1997|3     |
|25 |M     |writer    |Children|1994|1     |
|28 |M     |technician|Drama   |1994|2     |
|47 |M     |educator  |Crime   |1997|1     |
+---+------+----------+--------+----+------+
only showing top 5 rows


In [ ]:
# Učitavamo train i test skup iz MovieLens dataset-a.
# Podaci su već podeljeni (u1.base i u1.test).
train_ratings = spark.read.csv(
    f"{dataset_path}/u1.base",
    sep="\t",
    inferSchema=True,
    header=False
).toDF("user_id", "movie_id", "rating", "timestamp")

test_ratings = spark.read.csv(
    f"{dataset_path}/u1.test",
    sep="\t",
    inferSchema=True,
    header=False
).toDF("user_id", "movie_id", "rating", "timestamp")


# Spajamo sa korisnicima i filmovima i biramo relevantne kolone.
# Izbacujemo redove bez godine radi konzistentnosti podataka.
train_data = train_ratings.join(users, on="user_id") \
    .join(movies_features.select("movie_id", "genre", "year"), on="movie_id") \
    .select("age", "gender", "occupation", "genre", "year", "rating") \
    .filter(col("year").isNotNull())

train_data.show(5, truncate=False)


# Pripremamo test skup na isti način kao train.
test_data = test_ratings.join(users, on="user_id") \
    .join(movies_features.select("movie_id", "genre", "year"), on="movie_id") \
    .select("age", "gender", "occupation", "genre", "year", "rating") \
    .filter(col("year").isNotNull())

test_data.show(5, truncate=False)

+---+------+----------+---------+----+------+
|age|gender|occupation|genre    |year|rating|
+---+------+----------+---------+----+------+
|24 |M     |technician|Animation|1995|5     |
|24 |M     |technician|Action   |1995|3     |
|24 |M     |technician|Thriller |1995|4     |
|24 |M     |technician|Action   |1995|3     |
|24 |M     |technician|Crime    |1995|3     |
+---+------+----------+---------+----+------+
only showing top 5 rows
+---+------+----------+------+----+------+
|age|gender|occupation|genre |year|rating|
+---+------+----------+------+----+------+
|24 |M     |technician|Drama |1995|5     |
|24 |M     |technician|Drama |1995|3     |
|24 |M     |technician|Crime |1995|5     |
|24 |M     |technician|Drama |1994|5     |
|24 |M     |technician|Action|1996|3     |
+---+------+----------+------+----+------+
only showing top 5 rows


Podaci se nalaze kao tekst ili brojevi,to nije pogodno za modele predvidjanja,moramo ih svesti na isti nivo.Ovo radimo da bi mašinski model mogao da koristi kategorijske i numeričke podatke u numeričkom formatu, jer ML algoritmi ne mogu direktno da rade sa tekstualnim vrednostima.

In [ ]:
# Import ML alata za obradu kategorijskih i numeričkih podataka
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline

# StringIndexer pretvara kategorijske vrednosti u numeričke indekse
gender_indexer = StringIndexer(inputCol="gender", outputCol="gender_index", handleInvalid="keep")
occupation_indexer = StringIndexer(inputCol="occupation", outputCol="occupation_index", handleInvalid="keep")
genre_indexer = StringIndexer(inputCol="genre", outputCol="genre_index", handleInvalid="keep")

# OneHotEncoder pretvara indekse u vektore (bez implicitnog redosleda kategorija)
encoder = OneHotEncoder(
    inputCols=["gender_index", "occupation_index", "genre_index"],
    outputCols=["gender_vec", "occupation_vec", "genre_vec"]
)

# Spajamo sve feature-e u jedan vektor koji koristi ML model
assembler = VectorAssembler(
    inputCols=["age", "year", "gender_vec", "occupation_vec", "genre_vec"],
    outputCol="features"
)

-Definiše se Linear Regression model za predikciju ocena.
-Pipeline povezuje obradu podataka i treniranje modela u jedan -proces.
-ParamGridBuilder testira različite hiperparametre modela.
-Evaluator : (RMSE / MAE)
MAE meri prosečnu apsolutnu grešku, dok RMSE više kažnjava velike greške zbog kvadriranja.

-CrossValidator bira najbolju kombinaciju parametara pomoću 3-fold validacije.
-Model se trenira, pravi predikcije i evaluira pomoću RMSE metrike.

Linearna regresija
Regresioni model koji predviđa numeričku vrednost kao linearnu kombinaciju feature-a:
y=w^Tx+w0
Dobra je za jednostavne i linearne odnose između podataka.
Random Forest
Ansambl više stabala odlučivanja koji pravi predikciju prosekom svih stabala. Dobro hvata nelinearne obrasce i smanjuje overfitting u odnosu na jedno stablo

In [ ]:
# Import modela, validacije i evaluacije
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import RegressionEvaluator

# Definišemo Linear Regression model
lr = LinearRegression(
    featuresCol="features",
    labelCol="rating"
)

# Pipeline: obrada podataka + model
lr_pipeline = Pipeline(stages=[
    gender_indexer,
    occupation_indexer,
    genre_indexer,
    encoder,
    assembler,
    lr
])

# Definišemo grid hiperparametara za tuning
lr_paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0.01, 0.1, 1.0]) \
    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0]) \
    .build()

# Evaluacija modela (RMSE)
evaluator = RegressionEvaluator(
    labelCol="rating",
    predictionCol="prediction",
    metricName="rmse"
)

# Cross-validation za izbor najboljeg modela
lr_cv = CrossValidator(
    estimator=lr_pipeline,
    estimatorParamMaps=lr_paramGrid,
    evaluator=evaluator,
    numFolds=3
)

# Treniranje modela
lr_cv_model = lr_cv.fit(train_data)

# Predikcije na test skupu
lr_predictions = lr_cv_model.transform(test_data)

# Računamo RMSE
lr_rmse = evaluator.evaluate(lr_predictions)
print("Linear Regression RMSE:", lr_rmse)

# Najbolji parametri
best_lr = lr_cv_model.bestModel.stages[-1]
print("Best regParam:", best_lr._java_obj.getRegParam())
print("Best elasticNetParam:", best_lr._java_obj.getElasticNetParam())

Linear Regression RMSE: 1.1094338270658766
Best regParam: 0.01
Best elasticNetParam: 0.0


In [ ]:
# Import Random Forest modela
from pyspark.ml.regression import RandomForestRegressor

# Definišemo Random Forest model
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="rating"
)

# Pipeline: obrada podataka + model
rf_pipeline = Pipeline(stages=[
    gender_indexer,
    occupation_indexer,
    genre_indexer,
    encoder,
    assembler,
    rf
])

# Grid hiperparametara
rf_paramGrid = ParamGridBuilder() \
    .addGrid(rf.numTrees, [20, 50]) \
    .addGrid(rf.maxDepth, [5, 10]) \
    .build()

# Cross-validation
rf_cv = CrossValidator(
    estimator=rf_pipeline,
    estimatorParamMaps=rf_paramGrid,
    evaluator=evaluator,
    numFolds=3
)

# Trening modela
rf_cv_model = rf_cv.fit(train_data)

# Predikcije
rf_predictions = rf_cv_model.transform(test_data)

# Evaluacija
rf_rmse = evaluator.evaluate(rf_predictions)
print("Random Forest RMSE:", rf_rmse)

# Najbolji model i parametri (ISPRAVLJENO)
best_rf = rf_cv_model.bestModel.stages[-1]

print("Best numTrees:", best_rf.getNumTrees)
print("Best maxDepth:", best_rf.getMaxDepth())

Random Forest RMSE: 1.078285728801363
Best numTrees: 50
Best maxDepth: 10


**Random Fores**t je ostvario bolji rezultat jer ima manji **RMSE** (1.078) u odnosu na Linear Regression (1.109). Manji RMSE znači manju prosečnu grešku predikcije.
Najbolji parametri za Random Forest:
numTrees = 50 → korišćeno je 50 stabala, što daje stabilnije predikcije.
maxDepth = 10 → stabla su dovoljno duboka da uhvate složenije obrasce bez prevelikog overfitting-a.
          Najbolji parametri za **Linear Regression**:

regParam = 0.01 → mala regularizacija, modelu nije trebalo jako ograničavanje.
elasticNetParam = 0.0 → korišćena je L2 regularizacija (Ridge).

Z**aključak:**
Random Forest bolje modeluje nelinearne odnose između korisnika i filmova, pa daje preciznije preporuke od Linear Regression modela.

- Napravite proceduru za preporuku na sledeći način:
o Ulaz: korisnik (age, gender, occupation), 50 najpopularnijih filmova određenih u
prethodnom zadatku (genre, year).
o Za datog korisnika predvideti ocene na osnovu modela sa najboljom performansom
o Izlaz: Rangirani filmovi

In [ ]:
# Kreiranje skupa od 50 najpopularnijih filmova sa informacijama o žanru i godini izlaska
top_50_movies = popular_movies.limit(50) \
    .join(movies_features.select("movie_id", "genre", "year"), on="movie_id") \
    .select("title", "genre", "year")

top_50_movies.show(5, truncate=False)

+-------------------------+---------+----+
|title                    |genre    |year|
+-------------------------+---------+----+
|Toy Story (1995)         |Animation|1995|
|Twelve Monkeys (1995)    |Drama    |1995|
|Dead Man Walking (1995)  |Drama    |1995|
|Mr. Holland's Opus (1995)|Drama    |1995|
|Braveheart (1995)        |Action   |1995|
+-------------------------+---------+----+
only showing top 5 rows


In [ ]:
# Procedura za preporuku filmova za zadatog korisnika
def recommend_movies(age, gender, occupation, top_movies_df):
    from pyspark.sql.functions import lit, round, col

    # Dodajemo podatke o korisniku svakom od top 50 filmova
    input_df = top_movies_df \
        .withColumn("age", lit(age)) \
        .withColumn("gender", lit(gender)) \
        .withColumn("occupation", lit(occupation))

    # Predviđamo ocene pomoću najboljeg modela
    predictions = rf_cv_model.transform(input_df)

    # Rangiramo filmove po predviđenoj oceni
    ranked = predictions.select(
        "title",
        "genre",
        "year",
        round(col("prediction"), 2).alias("predicted_rating")
    ).orderBy(col("predicted_rating").desc())

    return ranked

In [ ]:
# Unos podataka o korisniku za kog model predviđa ocene i generiše preporuke filmova
result = recommend_movies(
    age=25,
    gender="M",
    occupation="student",
    top_movies_df=top_50_movies
)

result.show(10, truncate=False)

+--------------------------------------+------+----+----------------+
|title                                 |genre |year|predicted_rating|
+--------------------------------------+------+----+----------------+
|Godfather, The (1972)                 |Action|1972|3.96            |
|Star Wars (1977)                      |Action|1977|3.91            |
|Alien (1979)                          |Action|1979|3.91            |
|Amadeus (1984)                        |Drama |1984|3.91            |
|Jaws (1975)                           |Action|1975|3.91            |
|Raiders of the Lost Ark (1981)        |Action|1981|3.9             |
|Monty Python and the Holy Grail (1974)|Comedy|1974|3.9             |
|Empire Strikes Back, The (1980)       |Action|1980|3.9             |
|Return of the Jedi (1983)             |Action|1983|3.82            |
|Terminator, The (1984)                |Action|1984|3.82            |
+--------------------------------------+------+----+----------------+
only showing top 10 

3. Sistem preporuke baziran na **kolaborativnom filtriranju**

- Koristeći skup podataka u1.base cross validacijom optimizovati hiperparametre ALS
modela. Performanse najboljeg modela proceniti na test skupu podataka (u1.test).
Diskutovati da li postoji pre-treniranje ili pod-treniranje modela.

**Colaborativno filtriranje** preporučuje stavke na osnovu sličnosti korisnika ili stavki, koristeći prethodne interakcije i ocene korisnika. Prednost je što može otkriti skrivene obrasce bez poznavanja karakteristika sadržaja, dok su mane cold start problem i velika sparsity user-item matrice. Korsiti** U-I** matricu.
**Matrix Factorization** razlaže user-item matricu na latentne faktore korisnika i filmova:
R≈U⋅VTR
Time se otkrivaju skrivene preference korisnika i poboljšava preporuka nad sparse podacima.

**ALS** algoritam iterativno o**ptimizuje latentne faktore korisnika i stavki minimizacijom greške između stvarnih i predviđenih ocena.** Koristi se jer je efikasan za velike i sparse recommender sisteme.

**User/Item similarity** preporuke rade tako što pronalaze **slične korisnike** ili slične filmove na osnovu ocena i preporučuju stavke sa najvećom sličnošću.

In [ ]:
# Implementacija ALS (Alternating Least Squares) kolaborativnog sistema preporuke.
# Učitavaju se trening i test podaci, definiše ALS model nad user-item matricom,
# zatim se pomoću Cross-Validation optimizuju hiperparametri (rank, regParam, maxIter).
# Model se trenira nad trening skupom, evaluira pomoću RMSE metrike na trening i test podacima,
# a na kraju se ispisuju najbolji parametri modela i performanse preporuke.
from pyspark.ml.recommendation import ALS
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import col

train_ratings = spark.read.csv(
    f"{dataset_path}/u1.base",
    sep="\t",
    inferSchema=True
).toDF("user_id", "movie_id", "rating", "timestamp")

test_ratings = spark.read.csv(
    f"{dataset_path}/u1.test",
    sep="\t",
    inferSchema=True
).toDF("user_id", "movie_id", "rating", "timestamp")

als = ALS(
    userCol="user_id",
    itemCol="movie_id",
    ratingCol="rating",
    coldStartStrategy="drop",
    nonnegative=True
)

als_paramGrid = ParamGridBuilder() \
    .addGrid(als.rank, [5, 10, 20]) \
    .addGrid(als.regParam, [0.1, 0.2, 0.5]) \
    .addGrid(als.maxIter, [5, 10]) \
    .build()

evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

als_cv = CrossValidator(
    estimator=als,
    estimatorParamMaps=als_paramGrid,
    evaluator=evaluator,
    numFolds=3
)

als_cv_model = als_cv.fit(train_ratings)

train_predictions = als_cv_model.transform(train_ratings)
train_rmse = evaluator.evaluate(train_predictions)
print("ALS Train RMSE:", train_rmse)

test_predictions = als_cv_model.transform(test_ratings)
test_rmse = evaluator.evaluate(test_predictions)
print("ALS Test RMSE:", test_rmse)

best_model = als_cv_model.bestModel
print("Best rank:", best_model.rank)
print("Best regParam:", best_model._java_obj.parent().getRegParam())
print("Best maxIter:", best_model._java_obj.parent().getMaxIter())

ALS Train RMSE: 0.8155214877476744
ALS Test RMSE: 0.9340878903714995
Best rank: 5
Best regParam: 0.1
Best maxIter: 10


Ako je train RMSE značajno manji od test RMSE, model pokazuje znake overfitting-a.
Ako su oba RMSE visoka i slična, model je underfitted.
Pošto su train_rmse i test_rmse približni, model generalizuje dobro i nema značajnog overfitting-a.

In [ ]:
rmse_gap = test_rmse - train_rmse
print(f"RMSE gap (test - train): {rmse_gap:.4f}")

if rmse_gap > 0.15:
    print("Zakljucak: model pokazuje znake overfitting-a.")
elif train_rmse > 1.2 and test_rmse > 1.2:
    print("Zakljucak: model je verovatno underfitted (visok train i test RMSE).")
else:
    print("Zakljucak: nema jakih znakova overfitting-a; generalizacija je solidna.")
    # Poređenje train i test RMSE vrednosti radi analize overfitting-a i sposobnosti generalizacije modela.

RMSE gap (test - train): 0.1186
Zakljucak: nema jakih znakova overfitting-a; generalizacija je solidna.


- Korišćenjem u2.base identifikujte korisnike koji su dali manje od 5 ocena ili se ne pojavljuju
u u1.base i napravite dve promenljive: users_with_grades (korisnici iz u2.base koji se
pojavljuju u u1.base i imaju minimum 5 ocena) i users_without_grades (korisnici iz u2.base
koji se ne pojavljuju u u1.base ili imaju manje od 5 ocena)). Takođe, odvojiti filmove u dve
različite promenljive: movies_u2_only – filmovi koji se ne nalaze u u1.base a nalaze se u
u2_base i movies_u_12 – filmovi koji se nalaze u obe tabele.

In [ ]:
# Analiza korisnika i filmova između skupova u1 i u2 radi izdvajanja poznatih i novih korisnika/stavki za preporuku.
# Korisnici sa dovoljno ocena mogu koristiti collaborative filtering (ALS)
# dok se za nove korisnike ili korisnike sa malo ocena rešava cold-start problem alternativnim pristupom.
from pyspark.sql.functions import count

u2_ratings = spark.read.csv(
    f"{dataset_path}/u2.base",
    sep="\t",
    inferSchema=True
).toDF("user_id", "movie_id", "rating", "timestamp")

u2_user_counts = u2_ratings.groupBy("user_id").agg(
    count("*").alias("num_ratings")
)

u1_users = train_ratings.select("user_id").distinct()

users_with_grades = u2_user_counts.join(
    u1_users,
    on="user_id",
    how="inner"
).filter("num_ratings >= 5")

users_without_grades = u2_user_counts.join(
    u1_users,
    on="user_id",
    how="left_anti"
).union(
    u2_user_counts.join(
        u1_users,
        on="user_id",
        how="inner"
    ).filter("num_ratings < 5")
)

u1_movies = train_ratings.select("movie_id").distinct()
u2_movies = u2_ratings.select("movie_id").distinct()

movies_u2_only = u2_movies.join(
    u1_movies,
    on="movie_id",
    how="left_anti"
)

movies_u_12 = u2_movies.join(
    u1_movies,
    on="movie_id",
    how="inner"
)

print("Users with grades:", users_with_grades.count())
print("Users without grades:", users_without_grades.count())

print("Movies only in u2:", movies_u2_only.count())
print("Movies in both:", movies_u_12.count())

Users with grades: 943
Users without grades: 0
Movies only in u2: 32
Movies in both: 1616


- Napraviti proceduru koja će deliti korisnike iz u2.base na osnovu prethodnog opisa i koja će
za users_with_grades previđati na osnovu ALS-a, a za users_without_grades predviđati na
osnovu regresionog modela iz prethodne stavke. Na izlazu procedure treba da postoje
predikcije za sve korisnike.


Ovo je hibridno resenje, korisnici koji imaju dovoljno ocena koriste ALS(Kolaborativno filtriranje) a oni koji nemaju korsitimo Content based to je zabravo ML

In [ ]:
# Kreiranje hibridnog sistema preporuke gde korisnici sa dovoljno ocena koriste ALS (Collaborative Filtering)
#dok se za korisnike sa malo ili bez ocena koristi regresioni Content-Based model radi rešavanja cold-start problema.

users_with_profiles = users_with_grades.select("user_id").join(
    users.select("user_id", "age", "gender", "occupation"),
    on="user_id",
    how="inner"
 )

users_without_profiles = users_without_grades.select("user_id").join(
    users.select("user_id", "age", "gender", "occupation"),
    on="user_id",
    how="inner"
 )

print("users_with_profiles:", users_with_profiles.count())
print("users_without_profiles:", users_without_profiles.count())

users_with_profiles: 943
users_without_profiles: 0


In [ ]:
#Funkcija za generisanje hibridnih preporuka: za korisnike sa dovoljno ocena koriste se ALS predikcije (Collaborative Filtering)
# dok se za nove korisnike ili korisnike sa malo ocena koriste predikcije regresionog Content-Based modela,
# nakon čega se sve preporuke spajaju u jedinstven rezultat.

def build_hybrid_predictions():
    empty_schema = "user_id INT, movie_id INT, prediction DOUBLE"

    if users_with_profiles.rdd.isEmpty():
        als_predictions = spark.createDataFrame([], empty_schema)
    else:
        als_input = users_with_profiles.select("user_id").distinct().crossJoin(
            movies_u_12.select("movie_id").distinct()
        )
        als_predictions = best_model.transform(als_input).select(
            "user_id", "movie_id", "prediction"
        )

    if users_without_profiles.rdd.isEmpty():
        reg_predictions = spark.createDataFrame([], empty_schema)
    else:
        reg_candidate_movies = (
            u2_movies.join(
                movies_features.select("movie_id", "genre", "year"),
                on="movie_id",
                how="inner"
            )
            .filter(col("year").isNotNull())
            .select("movie_id", "genre", "year")
            .distinct()
        )

        reg_input = users_without_profiles.crossJoin(reg_candidate_movies)
        reg_predictions = best_regression_model.transform(reg_input).select(
            "user_id", "movie_id", "prediction"
        )

    return als_predictions.unionByName(reg_predictions)

In [ ]:
# Pokretanje hibridnog sistema preporuke i prikaz ukupnog broja generisanih predikcija i korisnika za koje su preporuke uspešno kreirane.

all_hybrid_predictions = build_hybrid_predictions()

print("Total hybrid predictions:", all_hybrid_predictions.count())
print(
    "Distinct users covered:",
    all_hybrid_predictions.select("user_id").distinct().count()
 )

all_hybrid_predictions.show(10, truncate=False)

Total hybrid predictions: 1523888
Distinct users covered: 943
+-------+--------+------------------+
|user_id|movie_id|prediction        |
+-------+--------+------------------+
|148    |496     |3.6853911876678467|
|148    |463     |3.2124953269958496|
|148    |471     |3.2847442626953125|
|148    |833     |2.948333263397217 |
|148    |148     |3.1675527095794678|
|148    |1088    |2.675703525543213 |
|148    |1238    |2.715740919113159 |
|148    |1591    |1.7691700458526611|
|148    |1645    |4.495873928070068 |
|148    |1342    |2.5751004219055176|
+-------+--------+------------------+
only showing top 10 rows


- Primenite proceduru iz prethodnog opisa i prikažite preporuke za korisnika koji je imao
najviše ocena u u2.base.

In [ ]:
# Preporuke
u2_base = spark.read.csv(
    f"{dataset_path}/u2.base",
    sep="\t",
    inferSchema=True
).toDF("user_id", "movie_id", "rating", "timestamp")

u2_base.show(5)

+-------+--------+------+---------+
|user_id|movie_id|rating|timestamp|
+-------+--------+------+---------+
|      1|       3|     4|878542960|
|      1|       4|     3|876893119|
|      1|       5|     3|889751712|
|      1|       6|     5|887431973|
|      1|       7|     4|875071561|
+-------+--------+------+---------+
only showing top 5 rows


Dobijamo korsinika koji ima najvise ocena, tako sto radio group po user it i aggregaciju po broju ocena


In [ ]:
from pyspark.sql.functions import count, desc

top_user = u2_base.groupBy("user_id") \
    .agg(count("*").alias("num_ratings")) \
    .orderBy(desc("num_ratings")) \
    .first()

top_user_id = top_user["user_id"]

print("User with most ratings:", top_user_id)
print("Number of ratings:", top_user["num_ratings"])

User with most ratings: 655
Number of ratings: 669


In [ ]:
from pyspark.sql.functions import count, desc

top_user = u2_base.groupBy("user_id") \
    .agg(count("*").alias("num_ratings")) \
    .orderBy(desc("num_ratings")) \
    .first()

top_user_id = top_user["user_id"]

print("User with most ratings:", top_user_id)
print("Number of ratings:", top_user["num_ratings"])

User with most ratings: 655
Number of ratings: 669


In [ ]:
# Preuzimanje osnovnih informacija o korisniku (godine, pol i zanimanje) koje će se koristiti za generisanje
# personalizovanih preporuka.

top_user_info = users.filter(col("user_id") == top_user_id).select(
    "age", "gender", "occupation"
).first()

user_age = top_user_info["age"]
user_gender = top_user_info["gender"]
user_occupation = top_user_info["occupation"]

print(user_age, user_gender, user_occupation)

50 F healthcare


In [ ]:
#Kreiramo preopruku za usera sa najvise ocena iz u2.base
recommendations = all_hybrid_predictions.filter(col("user_id") == top_user_id)
print("Predictions for top user:", recommendations.count())

Predictions for top user: 1616


In [ ]:
final_recommendations = recommendations.join(
    movies.select("movie_id", "title"),
    on="movie_id",
    how="left"
).select(
    "movie_id",
    "title",
    "prediction"
).orderBy("prediction", ascending=False)

final_recommendations.show(10, truncate=False)
#Prikaz predikcija

+--------+--------------------------------------+------------------+
|movie_id|title                                 |prediction        |
+--------+--------------------------------------+------------------+
|1449    |Pather Panchali (1955)                |3.9400177001953125|
|1512    |World of Apu, The (Apur Sansar) (1959)|3.873359203338623 |
|1642    |Some Mother's Son (1996)              |3.837505340576172 |
|1636    |Brothers in Trouble (1995)            |3.812533378601074 |
|1650    |Butcher Boy, The (1998)               |3.812533378601074 |
|1651    |Spanish Prisoner, The (1997)          |3.812533378601074 |
|1645    |Butcher Boy, The (1998)               |3.812533378601074 |
|913     |Love and Death on Long Island (1997)  |3.812533378601074 |
|1368    |Mina Tannenbaum (1994)                |3.808778762817383 |
|1643    |Angel Baby (1995)                     |3.7842226028442383|
+--------+--------------------------------------+------------------+
only showing top 10 rows


In [ ]:
from google.colab import drive
drive.mount('/content/drive')